In [5]:
import pandas as pd

df = pd.read_csv('./data/colleges.csv')
print("Data loaded!")
print("Shape:", df.shape)
df.head()

Data loaded!
Shape: (20, 10)


,college_name,branch,city,cutoff_2023,cutoff_2022,cutoff_2021,fees,avg_package,hostel,rating
0,VJTI Mumbai,Computer Engineering,Mumbai,98.5,97.8,97.2,125000,800000,Yes,4.8
1,VJTI Mumbai,Information Technology,Mumbai,97.2,96.5,96.0,125000,750000,Yes,4.7
2,SPIT Mumbai,Computer Engineering,Mumbai,95.8,95.2,94.8,180000,700000,No,4.5
3,SPIT Mumbai,Information Technology,Mumbai,94.5,93.8,93.2,180000,650000,No,4.4
4,DJ Sanghvi Mumbai,Computer Engineering,Mumbai,94.2,93.5,92.8,220000,720000,No,4.4


In [6]:
def recommend_colleges(percentile, branch, city=None, max_fees=None):
    
    # Step 1 — Branch filter karo
    filtered = df[df['branch'] == branch]
    
    # Step 2 — Eligible colleges — cutoff se kam percentile wale
    filtered = filtered[filtered['cutoff_2023'] <= percentile]
    
    # Step 3 — City filter (optional)
    if city:
        filtered = filtered[filtered['city'] == city]
    
    # Step 4 — Fees filter (optional)
    if max_fees:
        filtered = filtered[filtered['fees'] <= max_fees]
    
    # Step 5 — Rating ke hisaab se sort karo
    filtered = filtered.sort_values('rating', ascending=False)
    
    return filtered[['college_name','branch','city',
                      'cutoff_2023','fees','avg_package','rating']]

# Test karo
result = recommend_colleges(
    percentile=95.0,
    branch='Computer Engineering',
    city='Mumbai'
)

print("Recommended Colleges:")
print(result)

Recommended Colleges:
         college_name                branch    city  cutoff_2023    fees  \
4   DJ Sanghvi Mumbai  Computer Engineering  Mumbai         94.2  220000   
18  KJ Somaiya Mumbai  Computer Engineering  Mumbai         92.8  285000   

    avg_package  rating  
4        720000     4.4  
18       650000     4.2  


In [7]:
def weighted_recommend(percentile, branch, city=None, 
                        max_fees=None, priority='placement'):
    
    # Basic filter
    filtered = df[df['branch'] == branch].copy()
    filtered = filtered[filtered['cutoff_2023'] <= percentile]
    
    if city:
        filtered = filtered[filtered['city'] == city]
    if max_fees:
        filtered = filtered[filtered['fees'] <= max_fees]
    
    if filtered.empty:
        print("Koi college nahi mila — criteria loosely karo!")
        return
    
    # Normalize karo 0-1 ke beech
    filtered['score_placement'] = filtered['avg_package'] / filtered['avg_package'].max()
    filtered['score_rating']    = filtered['rating'] / filtered['rating'].max()
    filtered['score_fees']      = 1 - (filtered['fees'] / filtered['fees'].max())
    
    # Weights — priority ke hisaab se
    if priority == 'placement':
        w1, w2, w3 = 0.5, 0.3, 0.2
    elif priority == 'fees':
        w1, w2, w3 = 0.2, 0.3, 0.5
    else:
        w1, w2, w3 = 0.33, 0.34, 0.33
    
    # Final score
    filtered['final_score'] = (
        w1 * filtered['score_placement'] +
        w2 * filtered['score_rating'] +
        w3 * filtered['score_fees']
    )
    
    # Sort by final score
    filtered = filtered.sort_values('final_score', ascending=False)
    
    print(f"\n🎯 Top Colleges for {branch} | Percentile: {percentile}")
    print(f"Priority: {priority}\n")
    
    for i, row in filtered.iterrows():
        print(f"✅ {row['college_name']}")
        print(f"   Cutoff: {row['cutoff_2023']} | Fees: ₹{row['fees']:,}")
        print(f"   Avg Package: ₹{row['avg_package']:,} | Score: {row['final_score']:.2f}")
        print()

# Test 1 — Placement priority
weighted_recommend(
    percentile=95.0,
    branch='Computer Engineering',
    city='Mumbai',
    priority='placement'
)

# Test 2 — Fees priority
weighted_recommend(
    percentile=95.0,
    branch='Computer Engineering',
    city='Mumbai',
    priority='fees'
)


🎯 Top Colleges for Computer Engineering | Percentile: 95.0
Priority: placement

✅ DJ Sanghvi Mumbai
   Cutoff: 94.2 | Fees: ₹220,000
   Avg Package: ₹720,000 | Score: 0.85

✅ KJ Somaiya Mumbai
   Cutoff: 92.8 | Fees: ₹285,000
   Avg Package: ₹650,000 | Score: 0.74


🎯 Top Colleges for Computer Engineering | Percentile: 95.0
Priority: fees

✅ DJ Sanghvi Mumbai
   Cutoff: 94.2 | Fees: ₹220,000
   Avg Package: ₹720,000 | Score: 0.61

✅ KJ Somaiya Mumbai
   Cutoff: 92.8 | Fees: ₹285,000
   Avg Package: ₹650,000 | Score: 0.47



In [8]:
import json

def get_recommendations_json(percentile, branch, 
                              city=None, max_fees=None, 
                              priority='placement'):
    
    filtered = df[df['branch'] == branch].copy()
    filtered = filtered[filtered['cutoff_2023'] <= percentile]
    
    if city:
        filtered = filtered[filtered['city'] == city]
    if max_fees:
        filtered = filtered[filtered['fees'] <= max_fees]
    
    if filtered.empty:
        return json.dumps({"error": "No colleges found"})
    
    filtered['score_placement'] = filtered['avg_package'] / filtered['avg_package'].max()
    filtered['score_rating']    = filtered['rating'] / filtered['rating'].max()
    filtered['score_fees']      = 1 - (filtered['fees'] / filtered['fees'].max())
    
    if priority == 'placement':
        w1, w2, w3 = 0.5, 0.3, 0.2
    elif priority == 'fees':
        w1, w2, w3 = 0.2, 0.3, 0.5
    else:
        w1, w2, w3 = 0.33, 0.34, 0.33
    
    filtered['final_score'] = (
        w1 * filtered['score_placement'] +
        w2 * filtered['score_rating'] +
        w3 * filtered['score_fees']
    )
    
    filtered = filtered.sort_values('final_score', ascending=False)
    
    # JSON format mein convert karo
    result = []
    for _, row in filtered.iterrows():
        result.append({
            "college_name": row['college_name'],
            "branch": row['branch'],
            "city": row['city'],
            "cutoff_2023": row['cutoff_2023'],
            "fees": int(row['fees']),
            "avg_package": int(row['avg_package']),
            "rating": row['rating'],
            "score": round(row['final_score'], 2)
        })
    
    return json.dumps(result, indent=2)

# Test karo
output = get_recommendations_json(
    percentile=95.0,
    branch='Computer Engineering',
    city='Mumbai',
    priority='placement'
)
print(output)

[
  {
    "college_name": "DJ Sanghvi Mumbai",
    "branch": "Computer Engineering",
    "city": "Mumbai",
    "cutoff_2023": 94.2,
    "fees": 220000,
    "avg_package": 720000,
    "rating": 4.4,
    "score": 0.85
  },
  {
    "college_name": "KJ Somaiya Mumbai",
    "branch": "Computer Engineering",
    "city": "Mumbai",
    "cutoff_2023": 92.8,
    "fees": 285000,
    "avg_package": 650000,
    "rating": 4.2,
    "score": 0.74
  }
]
